### Complete Preprocessing Script

In [ ]:
import numpy as np
import rasterio
import pandas as pd
import json
import gc
import tensorflow as tf

# ---------------------------------------------------------
# 1. MASTER CONSTANTS
# ---------------------------------------------------------
START_YEAR = 2018
END_YEAR = 2025
N_LAT, N_LON = 141, 231
CHANNELS = 6

# ---------------------------------------------------------
# PHASE 2.1: LOAD & MERGE
# ---------------------------------------------------------
print("Starting Phase 2.1: Loading & Merging 8 Years of Data...")
yearly_arrays = []

for year in range(START_YEAR, END_YEAR + 1):
    filename = f"Delhi_NCR_DataCube_{year}.tif"
    days_in_year = 366 if year % 4 == 0 else 365
    
    with rasterio.open(filename) as src:
        flat_data = src.read()
        reshaped = flat_data.reshape(days_in_year, CHANNELS, N_LAT, N_LON)
        keras_format = np.transpose(reshaped, (0, 2, 3, 1)).astype(np.float16)
        yearly_arrays.append(keras_format)
        print(f"  -> Loaded {year}")

master_data = np.concatenate(yearly_arrays, axis=0)
del yearly_arrays
gc.collect()
print(f"\nMaster DataCube Assembled! Total Shape: {master_data.shape}")

# ---------------------------------------------------------
# PHASE 2.2: HEALING & SCALING (BULLETPROOF FIX)
# ---------------------------------------------------------
print("\nStarting Phase 2.2: Healing NaNs & Min-Max Scaling...")
total_days = master_data.shape[0]

# Step A: Chemical Healing
for c in range(4):
    flat_channel = master_data[:, :, :, c].astype(np.float32).reshape(total_days, -1)
    df = pd.DataFrame(flat_channel)
    df = df.bfill().ffill() 
    master_data[:, :, :, c] = df.values.reshape(total_days, N_LAT, N_LON).astype(np.float16)

# Step B: Safe Min-Max Scaling
print("  -> Calculating Safe Min-Max Scalers...")
scalers = {}

for c in range(CHANNELS):
    # THE FIX: np.nanmin and np.nanmax ignore any stray NaNs from interpolation
    c_min = float(np.nanmin(master_data[:, :, :, c].astype(np.float32)))
    c_max = float(np.nanmax(master_data[:, :, :, c].astype(np.float32)))
    
    # If a channel is completely empty (which it shouldn't be), prevent division by zero
    if c_max == c_min:
        c_max = c_min + 1.0
        
    master_data[:, :, :, c] = (master_data[:, :, :, c] - c_min) / (c_max - c_min)
    scalers[f"Channel_{c}"] = {"min": c_min, "max": c_max}

with open("Scaling_Parameters.json", "w") as f:
    json.dump(scalers, f, indent=4)

# THE ULTIMATE SAFETY NET: Destroy any remaining NaNs globally before saving
print("  -> Final global scrub of rogue NaNs...")
master_data = np.nan_to_num(master_data, nan=0.0, posinf=1.0, neginf=0.0)

np.save("Delhi_NCR_Master_DataCube_Scaled.npy", master_data)
print("\nPhase 2.2 Complete! Data normalized and safely saved to disk.")
z

I0000 00:00:1776271747.461039   20809 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776271748.219169   20809 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/yashaswi-garg/anaconda3/envs/aerosense/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
I0000 00:00:1776271753.665958   20809 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will n

Starting Phase 2.1: Loading & Merging 8 Years of Data...
  -> Loaded 2018
  -> Loaded 2019
  -> Loaded 2020
  -> Loaded 2021
  -> Loaded 2022
  -> Loaded 2023
  -> Loaded 2024
  -> Loaded 2025

Master DataCube Assembled! Total Shape: (2922, 141, 231, 6)

Starting Phase 2.2: Healing NaNs & Min-Max Scaling...
  -> Calculating Safe Min-Max Scalers...
  -> Final global scrub of rogue NaNs...
